# Create Fondation des Treilles Prix Jeune Chercheur Awards

Creates OpenAlex award rows from the official Fondation des Treilles Prix jeune chercheur laureate table.

**Prerequisites:**
- Admin/human with AWS credentials runs `scripts/local/treilles_young_researcher_to_s3.py` to download the official Fondation des Treilles pages, write parquet, and upload it.
- Contractor has prepared the script, notebook, registry entry, tracker row, and local QA, but does not have S3/Databricks access.
- The downloader writes all parquet columns as strings per `plans/awards/how-to-add-a-funder.md`; this notebook does type conversion with `TRY_CAST` / `TRY_TO_DATE`.

**Data sources:** `https://www.fondationdestreilles.com/la-recherche/le-prix-jeune-chercheur/les-laureats-du-prix-jeune-chercheur/` and `https://www.fondationdestreilles.com/la-recherche/le-prix-jeune-chercheur/`  
**S3 location:** `s3a://openalex-ingest/awards/treilles/treilles_young_researcher.parquet`  
**Local full-source validation on 2026-05-28:** 612 Prix jeune chercheur rows, 1983-2026; 100% native ID/name/year/amount/currency/display/landing-page coverage; 611/612 rows include discipline; EUR 3,672,000 total.

**Fondation des Treilles funder:**
- funder_id: 4320327761
- display_name: "Fondation des Treilles"
- country: FR

**Source-shape note:** The source is an official static laureate table with name, discipline, and award year. One row has no discipline in the source and is preserved with NULL discipline; the source also includes one repeated English header row, which the downloader skips.

**Amount/currency decision:** The official program page states the Prix jeune chercheur amount is 6,000 euros. The laureate table does not publish per-row exceptions, so `amount=6000` and `currency='EUR'` are applied to each award with `amount_note` documenting the source rule.

**Mapping summary:** One OpenAlex award row per listed Prix jeune chercheur laureate/beneficiary. Native key is `treilles-young-researcher-{year}-{name_slug}-{source_hash}`. The laureate is mapped to `lead_investigator`; the source discipline is retained in the description and raw table rather than mapped as an affiliation.


## Step 1: Create Raw Table


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.treilles_young_researcher_raw
USING delta
AS
SELECT
    *,
    current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/treilles/treilles_young_researcher.parquet`;


In [ ]:
%sql
-- Check raw row count. Local validation on 2026-05-28 found 612 rows.
SELECT COUNT(*) as total_treilles_young_researcher_awards
FROM openalex.awards.treilles_young_researcher_raw;


In [ ]:
%sql
-- Verify actual column names before transform logic references them.
DESCRIBE openalex.awards.treilles_young_researcher_raw;


In [ ]:
%sql
-- Sample raw Fondation des Treilles data.
SELECT
    funder_award_id,
    name,
    given_name,
    family_name,
    discipline,
    award_year,
    amount,
    currency,
    landing_page_url
FROM openalex.awards.treilles_young_researcher_raw
ORDER BY TRY_CAST(award_year AS INT), name
LIMIT 25;


## Step 1.5: Inspect Raw Data, Money, Dates, and Native Keys


In [ ]:
%sql
-- Money-field scan per runbook Step 1.5.
SELECT column_name
FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'treilles_young_researcher_raw'
  AND LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|funding|cost|budget|obligated|outlay|grant|award|donation|support|stipend|prize';


In [ ]:
%sql
-- Confirm coverage before mapping.
SELECT
    COUNT(*) AS total,
    COUNT(funder_award_id) AS rows_with_native_id,
    COUNT(name) AS rows_with_name,
    COUNT(given_name) AS rows_with_given_name,
    ROUND(try_divide(COUNT(given_name), COUNT(*)) * 100.0, 1) AS pct_given_name,
    COUNT(family_name) AS rows_with_family_name,
    ROUND(try_divide(COUNT(family_name), COUNT(*)) * 100.0, 1) AS pct_family_name,
    COUNT(discipline) AS rows_with_discipline,
    ROUND(try_divide(COUNT(discipline), COUNT(*)) * 100.0, 1) AS pct_discipline,
    COUNT(amount) AS rows_with_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) AS total_amount_eur,
    MIN(TRY_CAST(award_year AS INT)) AS min_award_year,
    MAX(TRY_CAST(award_year AS INT)) AS max_award_year,
    collect_set(currency) AS currencies
FROM openalex.awards.treilles_young_researcher_raw;


In [ ]:
%sql
-- Native-key inspection. funder_award_id must be unique.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(DISTINCT source_hash) AS distinct_source_hashes,
    COUNT(DISTINCT name) AS distinct_names
FROM openalex.awards.treilles_young_researcher_raw;


In [ ]:
%sql
-- Year and discipline distribution.
SELECT
    award_year,
    discipline,
    COUNT(*) AS cnt
FROM openalex.awards.treilles_young_researcher_raw
GROUP BY award_year, discipline
ORDER BY TRY_CAST(award_year AS INT), cnt DESC, discipline;


In [ ]:
%sql
-- Confirm the source row with missing discipline is preserved as NULL rather than inferred.
SELECT source_row_number, name, award_year, discipline, landing_page_url
FROM openalex.awards.treilles_young_researcher_raw
WHERE discipline IS NULL OR TRIM(discipline) = '';


## Step 1.6: Funder Existence Fail-Fast


In [ ]:
%sql
-- Must return TRUE. If this fails, STOP: Fondation des Treilles is missing from openalex.common.funder.
SELECT assert_true(
    COUNT(*) = 1,
    'Expected exactly one openalex.common.funder row for Fondation des Treilles F4320327761'
) AS funder_row_exists
FROM openalex.common.funder
WHERE funder_id = 4320327761;


In [ ]:
%sql
-- Inspect the funder row used for the award struct.
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.common.funder
WHERE funder_id = 4320327761;


## Step 2: Transform to OpenAlex Award Schema


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.treilles_young_researcher_awards
USING delta
AS
WITH
treilles_funder AS (
    SELECT
        funder_id,
        display_name,
        ror_id,
        doi
    FROM openalex.common.funder
    WHERE funder_id = 4320327761
),

raw_prepared AS (
    SELECT
        *,
        LOWER(TRIM(funder_award_id)) AS native_award_id,
        TRY_CAST(amount AS DOUBLE) AS parsed_amount,
        CASE WHEN TRY_CAST(amount AS DOUBLE) IS NOT NULL THEN 'EUR' ELSE NULL END AS parsed_currency,
        TRY_TO_DATE(CONCAT(award_year, '-01-01'), 'yyyy-MM-dd') AS parsed_start_date,
        TRY_TO_DATE(CONCAT(award_year, '-12-31'), 'yyyy-MM-dd') AS parsed_end_date,
        TRY_CAST(award_year AS INT) AS parsed_award_year
    FROM openalex.awards.treilles_young_researcher_raw
    WHERE funder_award_id IS NOT NULL
      AND TRIM(funder_award_id) != ''
      AND display_name IS NOT NULL
      AND TRIM(display_name) != ''
      AND name IS NOT NULL
      AND TRIM(name) != ''
),

awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000 as id,

        TRIM(g.display_name) as display_name,

        CASE
            WHEN g.description IS NULL OR TRIM(g.description) = '' THEN NULL
            ELSE TRIM(g.description)
        END as description,

        f.funder_id,
        g.native_award_id as funder_award_id,

        g.parsed_amount as amount,
        g.parsed_currency as currency,

        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,

        'prize' as funding_type,
        'Prix jeune chercheur' as funder_scheme,

        'treilles_prix_jeune_chercheur' as provenance,

        g.parsed_start_date as start_date,
        g.parsed_end_date as end_date,
        g.parsed_award_year as start_year,
        g.parsed_award_year as end_year,

        struct(
            NULLIF(TRIM(g.given_name), '') as given_name,
            NULLIF(TRIM(g.family_name), '') as family_name,
            CAST(NULL AS STRING) as orcid,
            g.parsed_start_date as role_start,
            struct(
                CAST(NULL AS STRING) as name,
                CAST(NULL AS STRING) as country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
            ) as affiliation
        ) as lead_investigator,

        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,

        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,

        NULLIF(TRIM(g.landing_page_url), '') as landing_page_url,
        CAST(NULL AS STRING) as doi,

        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000) as works_api_url,

        current_timestamp() as created_date,
        current_timestamp() as updated_date

    FROM raw_prepared g
    CROSS JOIN treilles_funder f
)

SELECT * FROM awards_transformed;


## Step 3: Delete Previous Source Rows and Insert Priority 178


In [ ]:
%sql
-- Remove previous Fondation des Treilles Prix jeune chercheur data before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'treilles_prix_jeune_chercheur' AND priority = 178;

-- Insert into openalex_awards_raw with exact OpenAlex awards schema plus priority.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    178 as priority
FROM openalex.awards.treilles_young_researcher_awards;


## Step 6: Verification Queries


In [ ]:
%sql
SELECT COUNT(*) as total_treilles_young_researcher_awards
FROM openalex.awards.treilles_young_researcher_awards;


In [ ]:
%sql
-- Confirm the transformed table has the OpenAlex awards columns only.
DESCRIBE openalex.awards.treilles_young_researcher_awards;


In [ ]:
%sql
-- Sample transformed awards.
SELECT
    id,
    display_name,
    funder_award_id,
    funder_scheme,
    funding_type,
    amount,
    currency,
    start_year,
    start_date,
    end_date,
    lead_investigator.given_name AS given_name,
    lead_investigator.family_name AS family_name,
    landing_page_url
FROM openalex.awards.treilles_young_researcher_awards
ORDER BY start_year, display_name
LIMIT 50;


In [ ]:
%sql
-- Check ID and native-key uniqueness.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT id) AS distinct_openalex_ids,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids
FROM openalex.awards.treilles_young_researcher_awards;


In [ ]:
%sql
-- Check funder distribution. Should be only Fondation des Treilles.
SELECT funder.display_name, funder_id, COUNT(*) as cnt
FROM openalex.awards.treilles_young_researcher_awards
GROUP BY funder.display_name, funder_id
ORDER BY cnt DESC;


In [ ]:
%sql
-- Data completeness and amount coverage.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(currency) as has_currency,
    COUNT(start_date) as has_start_date,
    COUNT(end_date) as has_end_date,
    COUNT(landing_page_url) as has_url,
    COUNT(lead_investigator.given_name) as has_given_name,
    COUNT(lead_investigator.family_name) as has_family_name,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) as pct_with_amount,
    ROUND(SUM(amount), 0) as total_eur
FROM openalex.awards.treilles_young_researcher_awards;


In [ ]:
%sql
-- Amount/currency fail-fast check for this prize pattern.
SELECT
    COUNT(*) AS total,
    SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END) AS rows_with_positive_amount,
    ROUND(try_divide(SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END), COUNT(*)) * 100.0, 1) AS pct_positive_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    ROUND(MIN(amount), 0) AS min_amount,
    ROUND(MAX(amount), 0) AS max_amount,
    ROUND(SUM(amount), 0) AS total_amount
FROM openalex.awards.treilles_young_researcher_awards;


In [ ]:
%sql
-- Year distribution.
SELECT start_year, COUNT(*) AS cnt, ROUND(SUM(amount), 0) AS total_eur
FROM openalex.awards.treilles_young_researcher_awards
GROUP BY start_year
ORDER BY start_year;


In [ ]:
%sql
-- Verify rows inserted into the raw awards table at priority 178.
SELECT
    priority,
    provenance,
    COUNT(*) as cnt,
    COUNT(DISTINCT funder_award_id) as distinct_funder_award_ids,
    ROUND(SUM(amount), 0) as total_eur
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'treilles_prix_jeune_chercheur' AND priority = 178
GROUP BY priority, provenance;


## Handoff / Admin Notes

Contractor-complete handoff only. The contractor has no S3 or Databricks access, so an admin must:

1. Run `scripts/local/treilles_young_researcher_to_s3.py` to upload `s3://openalex-ingest/awards/treilles/treilles_young_researcher.parquet`.
2. Run this notebook on Databricks.
3. Run the Step 6 verification cells and QA the inserted `provenance='treilles_prix_jeune_chercheur'`, `priority=178` rows.
4. Mark the tracker row Complete after production API verification.

Do not add job YAML until an admin has run and verified the ingest.
